In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
# silver_with_borough o'qish
silver_with_borough = spark.read.table("silver_openaq_with_borough")
print(f"✅ O'qildi: {silver_with_borough.count():,} qator")

StatementMeta(, bb19c550-37f2-4542-be80-13c981691be3, 3, Finished, Available, Finished, False)

✅ O'qildi: 25,496 qator


In [7]:
from pyspark.sql import functions as F

# Zararli parametrlar
zararli_parametrlar = ["pm25", "pm10", "pm1", "um003", "no2", "nox", "co", "o3", "so2", "no"]

# ============================================================
# 1. POLLUTION HOTSPOTS BY LOCATION/TIME
# ============================================================
gold_hotspot = silver_with_borough \
    .filter(F.col("parameter").isin(zararli_parametrlar)) \
    .withColumn("sana", F.to_date("datetime_from")) \
    .groupBy("borough", "sana", "parameter") \
    .agg(
        F.round(F.avg("value"), 3).alias("avg_value"),
        F.round(F.max("value"), 3).alias("max_value"),
        F.round(F.min("value"), 3).alias("min_value"),
        F.count("*").alias("kuzatish_soni")
    ) \
    .orderBy(F.desc("avg_value"))

print("=== 1. HOTSPOT ===")
display(gold_hotspot.limit(20))

# Warehousega saqlash
gold_hotspot.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_lakehouse.to_gold.gold_location_time")



StatementMeta(, bb19c550-37f2-4542-be80-13c981691be3, 11, Finished, Available, Finished, False)

=== 1. HOTSPOT ===


SynapseWidget(Synapse.DataFrame, f227feae-ba66-4f08-a911-a386619b9d30)

In [3]:
taxi_silver = spark.read.format("delta").load(
    "abfss://first_workspace@onelake.dfs.fabric.microsoft.com/itransition_project.Lakehouse/Tables/dbo/taxi_with_borough"
)

taxi_silver.show(5)
taxi_silver.printSchema()

StatementMeta(, bb19c550-37f2-4542-be80-13c981691be3, 6, Finished, Available, Finished, False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+-----------+------------+----------------+-----------+-----------+---------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_duration_minutes|pickup_hour|pickup_month|pickup_dayofweek|pickup_date|location_id|  borough|           zone_name|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----

In [21]:
# ============================================================
# 2. HIGH-TRAFFIC vs AIR QUALITY KORRELYATSIYA
# ============================================================

# Taxi: borough + sana bo'yicha aggregate
taxi_agg = taxi_silver \
    .withColumn("sana", F.to_date("tpep_pickup_datetime")) \
    .groupBy("borough", "sana") \
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    )

# OpenAQ: sana ustuni qo'shish
openaq_daily = silver_with_borough \
    .filter(F.col("parameter").isin(zararli_parametrlar)) \
    .withColumn("sana", F.to_date("datetime_from"))

# Inner join
traffic_air = openaq_daily.join(
    taxi_agg,
    (openaq_daily["borough"] == taxi_agg["borough"]) &
    (openaq_daily["sana"]    == taxi_agg["sana"]),
    "inner"
).select(
    openaq_daily["borough"],
    openaq_daily["sana"],
    openaq_daily["sensor_id"],
    openaq_daily["parameter"],
    openaq_daily["value"],
    openaq_daily["unit"],
    openaq_daily["datetime_from"],
    openaq_daily["datetime_to"],
    openaq_daily["summary_min"],
    openaq_daily["summary_max"],
    openaq_daily["summary_sd"],
    openaq_daily["expected_count"],
    openaq_daily["observed_count"],
    openaq_daily["percent_complete"],
    openaq_daily["percent_coverage"],
    taxi_agg["trip_count"],
    taxi_agg["avg_fare"],
    taxi_agg["avg_duration"],
    taxi_agg["total_revenue"]
)

# Korrelyatsiya
gold_corr = traffic_air \
    .groupBy("borough", "sana", "parameter") \
    .agg(
        F.avg("trip_count").alias("avg_trips"),
        F.avg("value").alias("avg_pollution")
    )

print("=== 2. KORRELYATSIYA ===")
gold_corr.agg(
    F.corr("avg_trips", "avg_pollution").alias("korrelyatsiya")
).show()

# Borough bo'yicha
gold_corr \
    .groupBy("borough") \
    .agg(F.corr("avg_trips", "avg_pollution").alias("korrelyatsiya")) \
    .orderBy(F.desc("korrelyatsiya")) \
    .show()

# Warehousega saqlash
gold_corr.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("to_gold.gold_traffic_air_corr")

# print("✅ gold_traffic_air_corr saqlandi!")

StatementMeta(, bb19c550-37f2-4542-be80-13c981691be3, 25, Finished, Available, Finished, False)

=== 2. KORRELYATSIYA ===
+--------------------+
|       korrelyatsiya|
+--------------------+
|-0.06146460630619033|
+--------------------+

+-------------+--------------------+
|      borough|       korrelyatsiya|
+-------------+--------------------+
|        Bronx| 0.08044774926870958|
|       Queens| 0.06264629026679673|
|    Manhattan|0.032323615889081156|
|Staten Island|-0.03313614365986987|
|     Brooklyn|-0.04463005906243...|
|      Unknown|-0.07671120254946796|
+-------------+--------------------+



In [20]:
# ============================================================
# 3. SEASONAL AIR QUALITY VARIATIONS
# ============================================================
gold_seasonal = silver_with_borough \
    .filter(F.col("parameter").isin(zararli_parametrlar)) \
    .withColumn("oy", F.month("datetime_from")) \
    .withColumn("mavsum", F
        .when(F.month("datetime_from").isin(12, 1, 2), "Qish")
        .when(F.month("datetime_from").isin(3, 4, 5),  "Bahor")
        .when(F.month("datetime_from").isin(6, 7, 8),  "Yoz")
        .otherwise("Kuz")
    ) \
    .groupBy("mavsum", "oy", "borough", "parameter") \
    .agg(
        F.round(F.avg("value"), 3).alias("avg_value"),
        F.round(F.max("value"), 3).alias("max_value"),
        F.round(F.min("value"), 3).alias("min_value"),
        F.count("*").alias("kuzatish_soni")
    ) \
    .orderBy("mavsum", "oy", "borough", "parameter")

print("=== 3. SEASONAL ===")
display(gold_seasonal.limit(20))

# Warehousega saqlash
gold_seasonal.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("to_gold.gold_seasonal")

print("✅ gold_seasonal saqlandi!")

StatementMeta(, bb19c550-37f2-4542-be80-13c981691be3, 24, Finished, Available, Finished, False)

=== 3. SEASONAL ===


SynapseWidget(Synapse.DataFrame, 4cfef9aa-af53-4cd2-98b2-e5b99381735d)

✅ gold_seasonal saqlandi!
